# Strategy 14. Naive strategy with graph schema

Evaluating strategy 14 - naive approach with graph schema - on BioMix test-set. Using enhanced schema.


In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]


In [3]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")


    
from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [5]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

Loading from an extended biomix test-set

In [6]:
questions = pd.read_csv("../biomix/testset/biomix_true_false_selected_augmented.csv")

## Running template-based query

- 14b - enchanced schema


In [7]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph.refresh_schema()

normal_schema = graph.schema

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_49992/506362744.py:5: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {nam

Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `targetInModel`: STRING 
  - `targetInModelMgiId`: STRING 
  - `targetFromSourceId`: STRING 
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `modelPhenotypeLabel`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`

In [8]:
from langchain_core.prompts import PromptTemplate

system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

{schema}

"""


normal_schema_description = f"""\
This is graph schema:
--------------------------------------------
{normal_schema}
--------------------------------------------    
"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{enhanced_schema}
--------------------------------------------    
"""

system_prompt_normal_schema = system_prompt_generic.format(schema = normal_schema_description)
system_prompt_enhanced_schema = system_prompt_generic.format(schema = enhanced_schema_description)

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)

# Option 14b - enhanced schema

In [9]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage
import openai
from anthropic import Anthropic

# Define base models to test
models_without_effort = ["gpt-4o"]  # does not have reasoning effort
models_with_effort = ["gpt-5.2-2025-12-11", "claude-sonnet-4-5-20250929"]

# Define reasoning effort levels
reasoning_efforts = ["medium", "high"]

# Create model configurations with effort levels
# Format: "model_name|effort_level" (e.g., "gpt-4o|medium", "gpt-4o|high")
models = models_without_effort + [f"{model}|{effort}" for model in models_with_effort for effort in reasoning_efforts]

niter = 1

# Create todo list: [(model_with_effort, question), ...]
todo = [(m, r['text']) for _, r in questions.iterrows() for m in models for _ in range(niter)]

def run_llm_14b(model_with_effort, question):

    try:
        
        #Check if this is a model with effort level (gpt-4o stays the same)
        if "|" in model_with_effort:
            model_name, effort = model_with_effort.split("|")
        else:
            model_name = model_with_effort # For GPT-4o
            effort = None
        
        if model_name == 'gpt-4o':
            llm = ChatModel(model = model_name)
            user_prompt = user_template.invoke({"question": question})

            messages = [
                SystemMessage(content=system_prompt_enhanced_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
            return result.content
            
        # For OpenAI reasoningmodels
        elif model_name.startswith("gpt-5"):
            client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            
            user_prompt = user_template.invoke({"question": question})
            
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt_enhanced_schema},
                    {"role": "user", "content": user_prompt.text}
                ],
                reasoning_effort=effort
            )
            return response.choices[0].message.content
        
        # For Claude models with extended thinking
        elif model_name.startswith("claude"):
            client = Anthropic()
            
            user_prompt = user_template.invoke({"question": question})
            
            # Build messages - prefill thinking block for extended reasoning
            messages = [{"role": "user", "content": user_prompt.text}]
            
            
            response = client.messages.create(
                model=model_name,
                max_tokens=16000,  # Extended thinking may need more tokens
                system=system_prompt_enhanced_schema,
                messages=messages,
                thinking={
                    "type": "enabled",
                    "budget_tokens": 12000 if effort == "high" else 6000
                }
            )      
        
            # Extract text content (thinking blocks are separate)
            text_content = ""
            for block in response.content:
                if block.type == "text":
                    text_content += block.text
            
            return text_content
        
        else:
            raise ValueError(f"Unsupported model: {model_name}")
            
    except Exception as e:
        print(f"Error with {model_with_effort}: {e}")
        return None

In [10]:
llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_14b(llm_model, question))

Prompting LLM: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [2:21:24<00:00, 16.97s/it]


In [11]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_14b(llm_model, question)
        time.sleep(2)

In [12]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:  55%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                       | 273/500 [25:21<27:15,  7.21s/it]Transaction failed and will be retried in 2.1810720201452636s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Transaction failed and will be retried in 1.1969639957738525s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Querying graph:  55%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                 

In [13]:
results = process_results(todo, llm_answers, cypher_results)
with open("../Thinking_models_only/results/biomix14-evaluations_ThinkingModels.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

In [14]:
results_df = pd.DataFrame(results)
results_df["has_interaction"] = results_df["count"] > 0
results_df = results_df.merge(questions, left_on="question", right_on="text", how="left")
results_df = results_df.drop(columns=["text"])
results_df['direct'] = ~results_df['question'].str.contains("is not associated")
results_df["answer"] = results_df["direct"] == results_df["has_interaction"]

In [15]:
results_df.to_excel("../Thinking_models_only/results/biomix14-evaluations_ThinkingModels.xlsx", index=False)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error,has_interaction,label,direct,answer
0,gpt-4o,Polycythemia Vera is not associated with Gene ...,To verify whether Polycythemia Vera is associa...,[{'query': 'MATCH (d:DiseaseOrPhenotypicFeatur...,1,MATCH (d:DiseaseOrPhenotypicFeature {name: 'Po...,True,[],0.462807,0.0,NaN,False,False,False,True
1,gpt-5.2-2025-12-11|medium,Polycythemia Vera is not associated with Gene ...,```cypher\n// Check whether Polycythemia vera ...,[{'query': '// Check whether Polycythemia vera...,1,// Check whether Polycythemia vera is associat...,False,NaN,NaN,NaN,Function execution timed out,False,False,False,True
2,gpt-5.2-2025-12-11|high,Polycythemia Vera is not associated with Gene ...,```cypher\n// 1) (Optional) Find the Disease n...,[{'query': '// 1) (Optional) Find the Disease ...,3,// 1) (Optional) Find the Disease node(s) matc...,True,"[{'d.id': 'efo:0002429', 'd.name': 'polycythem...",0.377880,1.0,NaN,True,False,False,False
3,claude-sonnet-4-5-20250929|medium,Polycythemia Vera is not associated with Gene ...,"Based on your statement, I can help you verify...",[{'query': 'MATCH (d:Disease) WHERE toLower(d....,1,MATCH (d:Disease)\nWHERE toLower(d.name) CONTA...,True,"[{'disease': 'polycythemia vera', 'disease_id'...",0.275240,518.0,NaN,True,False,False,False
4,claude-sonnet-4-5-20250929|high,Polycythemia Vera is not associated with Gene ...,I'll help you write a Cypher query to check if...,[{'query': 'MATCH (d:Disease) WHERE d.name =~ ...,1,MATCH (d:Disease)\nWHERE d.name =~ '(?i).*poly...,True,"[{'Disease': 'polycythemia vera', 'DiseaseID':...",0.290692,518.0,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,gpt-4o,Smith-Lemli-Opitz Syndrome is associated with ...,To confirm the association between Smith-Lemli...,[{'query': 'MATCH (disease:DiseaseOrPhenotypic...,1,MATCH (disease:DiseaseOrPhenotypicFeature)-[:I...,True,[],0.114103,0.0,NaN,False,False,True,False
496,gpt-5.2-2025-12-11|medium,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\n// Gene–disease association evidenc...,[{'query': '// Gene–disease association eviden...,1,"// Gene–disease association evidence for ""Smit...",True,[],0.124503,0.0,NaN,False,False,True,False
497,gpt-5.2-2025-12-11|high,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\n// Check whether Smith-Lemli-Opitz ...,[{'query': '// Check whether Smith-Lemli-Opitz...,1,// Check whether Smith-Lemli-Opitz Syndrome ha...,True,[],0.149552,0.0,NaN,False,False,True,False
498,claude-sonnet-4-5-20250929|medium,Smith-Lemli-Opitz Syndrome is associated with ...,Based on your statement about Smith-Lemli-Opit...,[{'query': 'MATCH (d:Disease)-[:IS_PART_OF]->(...,3,MATCH (d:Disease)-[:IS_PART_OF]->(assoc:GeneTo...,True,[],0.109808,0.0,NaN,False,False,True,False


In [16]:
# Calculate the fraction of correct answers for each model
accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()
accuracy_df.columns = ['model', 'accuracy']
accuracy_df

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_49992/2029851061.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()


,model,accuracy
0,claude-sonnet-4-5-20250929|high,0.78
1,claude-sonnet-4-5-20250929|medium,0.78
2,gpt-4o,0.57
3,gpt-5.2-2025-12-11|high,0.66
4,gpt-5.2-2025-12-11|medium,0.70


In [17]:
# # Show raw LLM answers for each model
# print("="*60)
# print("RAW LLM ANSWERS")
# print("="*60)

# for idx, row in results_df[0:20].iterrows():
#     print(f"\nModel: {row['model']}")
#     print(f"Question: {row['question'][:80]}...")
#     print(f"LLM Answer:")
#     print("-"*60)
#     print(row['llm_answer'])
#     print("="*60)